In [1]:
import os
import json
import pandas as pd
from pathlib import Path

# 1. กำหนดโครงสร้างโฟลเดอร์ Output
DATA_DIR = Path(".")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

dq_log = []

# ---------------------------------------------------------
# Step 5.1 Extract & Profiling
# ---------------------------------------------------------
orders_jan = pd.read_csv('orders_2026_01.csv')
orders_feb = pd.read_csv('orders_2026_02.csv')
customers = pd.read_csv('customers_crm.csv')
products = pd.read_excel('product_master.xlsx')

with open('payments.json', 'r', encoding='utf-8') as f:
    payments_raw = json.load(f)
payments = pd.json_normalize(payments_raw)

# ---------------------------------------------------------
# Step 5.2 Schema Alignment & Combine Orders
# ---------------------------------------------------------
orders_feb_aligned = orders_feb.rename(columns={
    'ordered_at': 'order_date',
    'qty': 'quantity',
    'discount_pct': 'discount'
})

def parse_discount(val):
    if pd.isna(val):
        return val
    if isinstance(val, str):
        val = val.replace('%', '').strip()
        return float(val) / 100.0
    return float(val)

orders_feb_aligned['discount'] = orders_feb_aligned['discount'].apply(parse_discount)
orders_jan['discount'] = orders_jan['discount'].apply(parse_discount)

orders_jan['order_date'] = pd.to_datetime(orders_jan['order_date'], errors='coerce')
orders_feb_aligned['order_date'] = pd.to_datetime(orders_feb_aligned['order_date'], format='%d/%m/%Y %H:%M', errors='coerce')

orders_combined = pd.concat([orders_jan, orders_feb_aligned], ignore_index=True)

# ---------------------------------------------------------
# Step 5.3 Transform (Clean & Standardize)
# ---------------------------------------------------------
# 1) Deduplicate Orders (เก็บแถวล่าสุด keep='last')
orders_dedup = orders_combined.drop_duplicates(subset=['order_id'], keep='last').copy()

dq_log.append({
    'step': 'Deduplication',
    'entity': 'orders',
    'issue_description': 'Duplicate order_id found in raw combined orders',
    'affected_records_count': len(orders_combined) - len(orders_dedup),
    'action_taken': 'Kept last record per order_id'
})

# 2) Clean Customers
def clean_province(p):
    if pd.isna(p):
        return p
    p = str(p).strip()
    mapping = {
        'Chonburi': 'ชลบุรี',
        'Bangkok': 'กรุงเทพมหานคร',
        'กทม.': 'กรุงเทพมหานคร',
        'กทม': 'กรุงเทพมหานคร',
        'Chiang Mai': 'เชียงใหม่',
        'Phuket': 'ภูเก็ต',
        'Rayong': 'ระยอง',
        'ขอนเเก่น': 'ขอนแก่น' # แก้คำผิดสระเอสองตัว
    }
    return mapping.get(p, p)

customers_clean = customers.copy()
customers_clean['email'] = customers_clean['email'].astype(str).str.strip().str.lower()
customers_clean['province'] = customers_clean['province'].apply(clean_province)
customers_clean = customers_clean.drop_duplicates(subset=['customer_id'], keep='last').copy()

dq_log.append({
    'step': 'Standardization & Deduplication',
    'entity': 'customers',
    'issue_description': 'Unstandardized province/email and duplicate customer_ids',
    'affected_records_count': len(customers) - len(customers_clean),
    'action_taken': 'Trimmed/lowercased email, mapped province to standard Thai names, kept last record per customer_id'
})

# 3) Clean Products & Payments
products_clean = products.copy()
products_clean['product_id'] = products_clean['product_id'].astype(str).str.strip()
products_clean = products_clean.drop_duplicates(subset=['product_id'], keep='last').copy()

payments_clean = payments.copy()
payments_clean['order_id'] = payments_clean['order_id'].astype(str).str.strip()
payments_clean = payments_clean.drop_duplicates(subset=['order_id'], keep='last').copy()

# ---------------------------------------------------------
# Step 5.4 Integrate & Validate
# ---------------------------------------------------------
missing_cust_df = orders_dedup[~orders_dedup['customer_id'].isin(customers_clean['customer_id'])]
dq_log.append({
    'step': 'Referential Integrity Check',
    'entity': 'orders',
    'issue_description': 'customer_id not found in customers_crm master data',
    'affected_records_count': len(missing_cust_df),
    'action_taken': 'Excluded from fact_sales table'
})

missing_prod_df = orders_dedup[~orders_dedup['product_id'].isin(products_clean['product_id'])]
dq_log.append({
    'step': 'Referential Integrity Check',
    'entity': 'orders',
    'issue_description': 'product_id not found in product_master data',
    'affected_records_count': len(missing_prod_df),
    'action_taken': 'Excluded from fact_sales table'
})

invalid_qty_df = orders_dedup[orders_dedup['quantity'] <= 0]
dq_log.append({
    'step': 'Business Rule Validation',
    'entity': 'orders',
    'issue_description': 'quantity <= 0',
    'affected_records_count': len(invalid_qty_df),
    'action_taken': 'Excluded from fact_sales table'
})

# Merge Orders with Payments
merged_df = pd.merge(orders_dedup, payments_clean, on='order_id', how='left', indicator='_merge_payment')

non_paid_df = merged_df[merged_df['payment.status'] != 'PAID']
dq_log.append({
    'step': 'Business Rule Validation',
    'entity': 'payments',
    'issue_description': 'payment.status is not PAID (FAILED, REFUNDED, or Missing)',
    'affected_records_count': len(non_paid_df),
    'action_taken': 'Excluded from fact_sales table'
})

# สร้าง Fact Sales (เฉพาะคำสั่งซื้อที่ผ่านเงื่อนไขธุรกิจทั้งหมด)
valid_orders_mask = (
    (merged_df['quantity'] > 0) &
    (merged_df['unit_price'] > 0) &
    (merged_df['discount'] >= 0) & (merged_df['discount'] <= 1) &
    (merged_df['customer_id'].isin(customers_clean['customer_id'])) &
    (merged_df['product_id'].isin(products_clean['product_id'])) &
    (merged_df['payment.status'] == 'PAID')
)

fact_sales = merged_df[valid_orders_mask].copy()
fact_sales['net_sales'] = fact_sales['quantity'] * fact_sales['unit_price'] * (1 - fact_sales['discount'])

# ---------------------------------------------------------
# Step 5.5 Load & Export Files
# ---------------------------------------------------------
dim_customer = customers_clean[['customer_id', 'full_name', 'email', 'province', 'signup_date']]
dim_product = products_clean[['product_id', 'product_name', 'category', 'standard_price', 'active_flag']]

fact_sales_out = fact_sales[[
    'order_id', 'order_date', 'customer_id', 'product_id',
    'quantity', 'unit_price', 'discount', 'channel',
    'payment_id', 'payment.method', 'payment.status', 'net_sales'
]].copy()

# Summaries
fact_sales_cust = pd.merge(fact_sales, customers_clean[['customer_id', 'province']], on='customer_id', how='left')
summary_by_province = fact_sales_cust.groupby('province', as_index=False)['net_sales'].agg(
    total_net_sales='sum',
    transaction_count='count'
).sort_values(by='total_net_sales', ascending=False)

fact_sales_prod = pd.merge(fact_sales, products_clean[['product_id', 'category']], on='product_id', how='left')
summary_by_category = fact_sales_prod.groupby('category', as_index=False)['net_sales'].agg(
    total_net_sales='sum',
    transaction_count='count'
).sort_values(by='total_net_sales', ascending=False)

data_quality_report = pd.DataFrame(dq_log)

# เซฟลงโฟลเดอร์ output/
dim_customer.to_csv(OUTPUT_DIR / 'dim_customer.csv', index=False, encoding='utf-8-sig')
dim_product.to_csv(OUTPUT_DIR / 'dim_product.csv', index=False, encoding='utf-8-sig')
fact_sales_out.to_csv(OUTPUT_DIR / 'fact_sales.csv', index=False, encoding='utf-8-sig')
data_quality_report.to_csv(OUTPUT_DIR / 'data_quality_report.csv', index=False, encoding='utf-8-sig')
summary_by_province.to_csv(OUTPUT_DIR / 'summary_by_province.csv', index=False, encoding='utf-8-sig')
summary_by_category.to_csv(OUTPUT_DIR / 'summary_by_category.csv', index=False, encoding='utf-8-sig')

print("ETL Pipeline completed successfully! 6 files generated in 'output/'.")

ETL Pipeline completed successfully! 6 files generated in 'output/'.
